- notebook for preprocessing downloaded pretraining data
- to download it, use dataset_constructor.py

In [ ]:
import json
from dataset_constructor import get_sld
from pathlib import Path
import pandas as pd
import numpy as np


import sys, os
data_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if data_root not in sys.path:
    sys.path.insert(0, data_root)

from utils.general import external_shuffle_csv, count_slds

In [ ]:
CHUNK_SIZE = 1_000_000

In [ ]:
count_slds(input_csv_path="pretraining_dataset.csv", output_json_path="domain_counts.json")

# SLD overlap calculation

In [ ]:
with open("domain_counts.json") as f: 
    current_pretrianing_domains = json.load(f)
    current_pretrianing_domains_set = set(current_pretrianing_domains)

In [ ]:
with open("../public_dataset/domain_counts.json")  as f: 
    public_domains = json.load(f)
    public_domains_set = set(public_domains)

In [ ]:
with open("../private_dataset/domain_counts_stratified_version.json") as f: private_domains_stratified = json.load(f)
with open("../private_dataset/domain_counts.json") as f: private_domains = json.load(f)
private_domain_set = set(private_domains) | set(private_domains_stratified)

In [ ]:
total_domains = len(current_pretrianing_domains_set | private_domain_set | public_domains_set)

common_domains = current_pretrianing_domains_set & (private_domain_set | public_domains_set)
intersect_count = len(common_domains)

domains_to_remove = intersect_count

urls_to_remove = sum(current_pretrianing_domains[dom] for dom in common_domains)

total_urls_first = sum(current_pretrianing_domains.values())
urls_remaining = total_urls_first - urls_to_remove

print(f"Total unique domains:      {total_domains}")
print(f"Common (intersection):     {intersect_count}")
print(f"Domains to drop from #1:   {domains_to_remove}")
print(f"URLs to drop from #1:      {urls_to_remove}")
print(f"URLs remaining in first:   {urls_remaining}")
print(f"Overlap: ({urls_to_remove / total_urls_first * 100}%)")

- remove URLs that have same SLD as sth from public or private finetuning dataset

In [ ]:
INPUT_CSV = Path("pretraining_dataset.csv")
OUTPUT_CSV = Path("pretraining_dataset_pruned.csv")
CHUNKSIZE = 1_000_000

first_chunk = True
reader = pd.read_csv(INPUT_CSV, chunksize=CHUNKSIZE)

for df in reader:
    df["sld"] = df["url"].map(get_sld)
    df = df[~df["sld"].isin(common_domains)]
    df = df.drop(columns=["sld"])
    df.to_csv(OUTPUT_CSV, mode="w" if first_chunk else "a", index=False, header=first_chunk)
    first_chunk = False

print(f"Done: written filtered data to {OUTPUT_CSV}")

# Remove duplicates
- run utils/remove_duplicates.cpp on it

In [ ]:
external_shuffle_csv("pretraining_dataset_pruned.csv", "pretraining_dataset_shuffled.csv")

# Random HTTP removal

In [ ]:
INPUT_FILE = Path("pretraining_dataset_shuffled.csv")
OUTPUT_FILE = Path("pretraining_dataset_http_preprocessed.csv")
HTTP_PROB = 0.10
HTTPS_PROB = 0.40


def process_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    urls = chunk["url"]

    # Boolean masks for URLs that start with each protocol
    http_mask = urls.str.startswith("http://")
    https_mask = urls.str.startswith("https://")

    # Generate independent random numbers for every row
    rand_http = np.random.rand(len(chunk))
    rand_https = np.random.rand(len(chunk))

    # Decide which rows lose their prefix
    strip_http = http_mask & (rand_http < HTTP_PROB)
    strip_https = https_mask & (rand_https < HTTPS_PROB)

    # Apply the replacements in‑place, using vectorised .str.replace
    chunk.loc[strip_http, "url"] = chunk.loc[strip_http, "url"].str.replace(r"^http://", "", regex=True)
    chunk.loc[strip_https, "url"] = chunk.loc[strip_https, "url"].str.replace(r"^https://", "", regex=True)
    return chunk

In [ ]:
first = True
if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

for chunk in pd.read_csv(
    INPUT_FILE,
    chunksize=CHUNK_SIZE,
    dtype={"url": "string"},
    keep_default_na=False
):
    processed = process_chunk(chunk)

    processed.to_csv(
        OUTPUT_FILE,
        mode="w" if first else "a",
        header=first,
        index=False,
    )
    first = False